In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## 1.INTRODUCTION

<font color="green">
Logistic regression is a supervised machine learning algorithm that predicts the probability, ranging from 0 to 1, of a datapoint belonging to a specific category, or class. These probabilities can then be used to assign, or classify, observations to the more probable group.

For example, we could use a logistic regression model to predict the probability that an incoming email is spam. If that probability is greater than 0.5, we could automatically send it to a spam folder. This is called binary classification because there are only two groups (eg., spam or not spam).

Some other examples of problems that we could solve using logistic regression:

Disease identification — Is a tumor malignant?
Customer conversion — Will a customer arriving on a sign-up page enroll in a service

<font color="green">
Linear regression model range from negative to positive infinity. Therefore, predictions of linear regression don’t really make sense for a classification problem. 

To build a logistic regression model, we apply a logit link function to the left-hand side of our linear regression function.
    
Notice that the red line stays between 0 and 1 on the y-axis. It now makes sense to interpret this value as a probability of group membership; whereas that would have been non-sensical for regular linear regression.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("darkgrid")
plt.figure(figsize=(12,10))
plt.imshow(plt.imread("../input/logistic-reg/Capture.PNG"))

<font color="green">
We replace y with the letter p because we are going to interpret it as a probability which is called log odds. The whole left-hand side of this equation is called log-odds because it is the natural logarithm (ln) of odds (p/(1-p)). The right-hand side of this equation looks exactly like regular linear regression

In [ ]:
plt.figure(figsize=(12,10))
plt.imshow(plt.imread("../input/logodss/Capture.PNG"))

<font color="green">
For example, suppose that the probability a student passes an exam is 0.7. That means the probability of failing is 1 - 0.7 = 0.3. Thus, the odds of passing are:0.7/0.3 = 2.33.This means that students are 2.33 times more likely to pass than to fail.

Odds can only be a positive number. When we take the natural log of odds (the log odds), we transform the odds from a positive value to a number between negative and positive infinity — which is exactly what we need! The logit function (log odds) transforms a probability (which is a number between 0 and 1) into a continuous value that can be positive or negative.

<font color="green">
Lets make some examples:Suppose that there is a 40% probability of rain today (p = 0.4). 
    
Suppose that there is a 90% probability that my train to work arrives on-time.

In [ ]:
# Calculate odds_of_rain
odds_of_rain = .4/.6
print('odds of rain: ', odds_of_rain)

# Calculate log_odds_of_rain
log_odds_of_rain = np.log(.4/.6)
print('log odds of rain: ', log_odds_of_rain)

odds_on_time = 0.9/0.1
print("odds of time: ", odds_on_time)

log_odds_on_time = np.log(odds_on_time)
print("log odds of rain: ",log_odds_on_time )

<font color="green">
In the first example with 40% probability, we get odds of rains lesser than 1 which means that it is more likely not to rain.
   
In the second example with 90% probability,we get odds of rains higher than 1 which means that it is 9 times more likely to rain.

<font color="green">
These predictions range from negative to positive infinity: these are log odds. Therefore, in order to get a probability between 0 and 1 , we send the result og log odds to Sigmoid Function.

In [ ]:
plt.figure(figsize=(12,10))
plt.imshow(plt.imread("../input/sigmoud-ffun/Capture.PNG"))

In [ ]:
x= 3.28
np.exp(x)/(np.exp(x)+1)

In [ ]:
def calculate_sigmoid(log_odds):
    return np.exp(log_odds)/(np.exp(log_odds)+1)

In [ ]:
print(calculate_sigmoid(log_odds_of_rain))
print(calculate_sigmoid(log_odds_on_time))

## 2. Logistic Regression Model in Sklearn

## 2.1. Exploratory Data Analysis

In [ ]:
df = pd.read_csv("../input/heart-disease-prediction-using-logistic-regression/framingham.csv")
df.head()

<font color="green">
Demographic:
    
• Sex: male or female(Nominal)
• Age: Age of the patient;(Continuous - Although the recorded ages have been truncated to whole numbers, the concept of age is continuous)
Behavioral
• Current Smoker: whether or not the patient is a current smoker (Nominal)
• Cigs Per Day: the number of cigarettes that the person smoked on average in one day.(can be considered continuous as one can have any number of cigarettes, even half a cigarette.)
    
Medical( history)
• BP Meds: whether or not the patient was on blood pressure medication (Nominal)
• Prevalent Stroke: whether or not the patient had previously had a stroke (Nominal)
• Prevalent Hyp: whether or not the patient was hypertensive (Nominal)
• Diabetes: whether or not the patient had diabetes (Nominal)
    
Medical(current)
• Tot Chol: total cholesterol level (Continuous)
• Sys BP: systolic blood pressure (Continuous)
• Dia BP: diastolic blood pressure (Continuous)
• BMI: Body Mass Index (Continuous)
• Heart Rate: heart rate (Continuous - In medical research, variables such as heart rate though in fact discrete, yet are considered continuous because of large number of possible values.)
• Glucose: glucose level (Continuous)
Predict variable (desired target)
    
• 10 year risk of coronary heart disease CHD (binary: “1”, means “Yes”, “0” means “No”)

In [ ]:
df.info()# all values are numerical

In [ ]:
df.describe(include="all").transpose()

In [ ]:
df.corr()["TenYearCHD"].sort_values(ascending=False)
#Here we can see the correlations between features and hearth disease

<font color="red">
According to these values above and in the figure below,people will be tend to hearth disease if
    
    they are older                
    they have higher systolic blood pressure 
    they are hypertensive 
    they have higher diastolic blood pressure
    they have higher glucose level glucose           
    they have diabetes           
    they are males               
    
    

In [ ]:
#Lets visualize overall correlations between all columns with each other
plt.figure(figsize=(25,20))
mask = np.zeros_like(df.corr(), dtype=np.bool)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(df.corr(),cmap="jet",annot=True,linewidths=0, linecolor='white',cbar=True,mask=mask)

## 2.2. Data Visualization

In [ ]:
labels = 'Heart Disease', 'Healthy'
sizes = [df.TenYearCHD[df["TenYearCHD"]==1].count(), df.TenYearCHD[df["TenYearCHD"]==0].count()]
explode = (0, 0.1)
fig1, ax1 = plt.subplots(figsize=(10, 8))
ax1.pie(sizes, explode=explode, labels=labels, autopct='%1.1f%%',
        shadow=True, startangle=90)
ax1.axis('equal')
plt.title("Proportion of Heart Disease and Healthy", size = 20)
plt.show()

<font color="green">
So about 15.2% of the people have disease while 84.8% have have not. So the baseline model could be to predict that 15.2% of people have disease. This means that we have unbalanced target which can affect the performance of the machine learning algorithm and its predictions negatively if we do not deal with this issue.

In [ ]:
plt.figure(figsize=(15,10))
sns.countplot(x="male",hue="TenYearCHD", data=df, palette="Set1")
plt.title("The Counts of Hearth Disease by Male")
plt.legend()

In [ ]:
plt.figure(figsize=(15,10))
sns.countplot(x="prevalentHyp", hue="TenYearCHD", data=df,palette="Set2")
plt.title("The Counts of Hearth Disease by systolic blood pressure ")

In [ ]:
import plotly.graph_objects as go
labels = df['education'].unique()
values = df['education'].value_counts()

fig = go.Figure(data=[go.Pie(labels=labels, values=values, hole=.3)])
fig.update_layout(title_text="<b>Education</b>")
fig.show()

In [ ]:
sns.set_style("darkgrid")
sns.kdeplot(x="diaBP",data=df,palette="Set2",hue="TenYearCHD",shade=True)
sns.set(rc={'figure.figsize':(20,12)})
plt.ylabel('Density');
plt.xlabel('Diastolic blood pressure');
plt.title('Distribution of  diastolic blood pressure by Heart Disease')

In [ ]:
sns.set_style("darkgrid")
sns.kdeplot(x="sysBP",data=df,palette="Set2",hue="TenYearCHD",shade=True)
sns.set(rc={'figure.figsize':(20,12)})
plt.ylabel('Density');
plt.xlabel('Systolic blood pressure ');
plt.title('Distribution of  systolic blood pressure  by Heart Disease')

## 2.3 Data Preprocessing

<font color="green">
1.Dealing with Missing Values

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
missing_columns = ["glucose", "education","BPMeds","totChol","cigsPerDay","BMI","heartRate"]
for i in missing_columns:
    print("Column :", i)
    print(df[i].nunique())
    print("******************************")

<font color="green">
We will replace columns with continuous values with mean while columns with discrete values with median

In [ ]:
df["glucose"].fillna(df["glucose"].mean(),inplace =True)
df["totChol"].fillna(df["totChol"].mean(),inplace=True)
df["cigsPerDay"].fillna(df["cigsPerDay"].mean(),inplace=True)
df["BMI"].fillna(df["BMI"].mean(),inplace=True)
df["heartRate"].fillna(df["heartRate"].mean(),inplace=True)
df["education"].fillna(df["education"].median(),inplace=True)
df["BPMeds"].fillna(df["BPMeds"].median(),inplace=True)

In [ ]:
df.isnull().sum().sort_values(ascending=False)

<font color="green">
2.Data Normalization

In [ ]:
from sklearn.preprocessing import StandardScaler
ss = StandardScaler()
df[["cigsPerDay","totChol","sysBP","diaBP","BMI","heartRate","glucose","age"]]= ss.fit_transform(df[["cigsPerDay","totChol","sysBP","diaBP","BMI","heartRate","glucose","age"]])
df.head(3)

<font color="green">
3.Splitting Data into Training and Test Set

In [ ]:
X = df.drop("TenYearCHD",axis=1)
y = df["TenYearCHD"]
print(X.columns)
print(y)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

## 3. Model Training with Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(X_train, y_train)
print(model.coef_)
print(model.intercept_)


In [ ]:
predictions = model.predict(X_test)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
print(confusion_matrix(y_test,predictions))
print(classification_report(y_test, predictions))
print(accuracy_score(y_test, predictions))

## 4. Using Other Classification Algorithms

## 4.1. Random Forests

In [ ]:
from sklearn.ensemble import RandomForestClassifier
forest = RandomForestClassifier()
forest.fit(X_train, y_train)
predictions2 = forest.predict(X_test)
print(confusion_matrix(y_test,predictions2))
print(classification_report(y_test, predictions2))
print(accuracy_score(y_test, predictions2))

In [ ]:
#Choosing best hyperparameters:
# Number of trees in random forest
n_estimators = [int(x) for x in np.linspace(start = 100, stop = 1200, num = 12)]
# Number of features to consider at every split
max_features = ['auto', 'sqrt']
# Maximum number of levels in tree
max_depth = [int(x) for x in np.linspace(5, 30, num = 6)]
# max_depth.append(None)
# Minimum number of samples required to split a node
min_samples_split = [2, 5, 10, 15, 100]
# Minimum number of samples required at each leaf node
min_samples_leaf = [1, 2, 5, 10]

In [ ]:
# Create the random grid for these hyperparameters:
random_grid = {'n_estimators': n_estimators,
               'max_features': max_features,
               'max_depth': max_depth,
               'min_samples_split': min_samples_split,
               'min_samples_leaf': min_samples_leaf}

print(random_grid)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
forest = RandomForestClassifier()
random_grid = RandomizedSearchCV(estimator = forest, param_distributions = random_grid, n_iter = 10, cv = 5, verbose=2, random_state=42, n_jobs = 1)

In [ ]:
random_grid.fit(X_train,y_train)

In [ ]:
print(random_grid.best_params_)
print(random_grid.best_score_)

In [ ]:
predictions3 = random_grid.predict(X_test)

In [ ]:

print(confusion_matrix(y_test,predictions3))
print(classification_report(y_test, predictions3))
print(accuracy_score(y_test, predictions3))

## 4.2. K Neighbors Classifier

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
error_rate=list()
#here we iterate meny different k values and plot their error rates 
#and discover which one is better than others and has the lowest error rate
for i in range(1,40):
    knn=KNeighborsClassifier(n_neighbors=i)
    knn.fit(X_train,y_train)
    prediction_i=knn.predict(X_test)
    error_rate.append(np.mean(prediction_i != y_test))

In [ ]:
# Now we will plot the prediction error rates of different k values
plt.figure(figsize=(15,10))
plt.plot(range(1,40),error_rate, color="blue", linestyle="--",marker="o",markerfacecolor="red",markersize=10)
plt.title("Error Rate vs K Value")
plt.xlabel="K Value"
plt.ylabel("Error Rate")

In [ ]:
knn=KNeighborsClassifier(n_neighbors=27)
knn.fit(X_train, y_train)
predictions4=knn.predict(X_test)
print(confusion_matrix(y_test,predictions4))
print(classification_report(y_test, predictions4))
print(accuracy_score(y_test, predictions4))

## 4.2 Support Vector Machines

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
param_grid={"C":[1,2,3,4,5,10,100],"gamma":[1,0.1,0.2,0.5,0.01,0.001,0.0001]} 
#here we select values for grid search to try
grid=GridSearchCV(SVC(),param_grid,verbose=2)
grid.fit(X_train,y_train)

In [ ]:
print(grid.best_params_)
print(grid.best_estimator_)

In [ ]:
grid_predictions=grid.predict(X_test)
print(confusion_matrix(y_test,grid_predictions))
print(classification_report(y_test, grid_predictions))
print(accuracy_score(y_test, grid_predictions))

## 4.4. Voting Classifiers

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import VotingClassifier
clf1 = KNeighborsClassifier(n_neighbors=27)
clf2 = RandomForestClassifier(n_estimators= 700, min_samples_split = 15, min_samples_leaf = 1,max_features= 'auto',max_depth = 20)
clf3 = AdaBoostClassifier()
eclf1 = VotingClassifier(estimators=[('knc', clf1), ('rfc', clf2), ('abc', clf3)], voting='soft')
eclf1.fit(X_train, y_train)
predictions = eclf1.predict(X_test)
print("Final Accuracy Score ")
print(confusion_matrix(y_test,grid_predictions))
print(classification_report(y_test, grid_predictions))
print(accuracy_score(y_test, grid_predictions))

In [ ]:
clf1 = GradientBoostingClassifier()
clf2 = LogisticRegression()
clf3 = AdaBoostClassifier()
eclf1 = VotingClassifier(estimators=[('gbc', clf1), ('lr', clf2), ('abc', clf3)], voting='soft')
eclf1.fit(X_train, y_train)
predictions = eclf1.predict(X_test)
print("Final Accuracy Score ")
print(accuracy_score(y_test, predictions))
print(confusion_matrix(y_test,grid_predictions))
print(classification_report(y_test, grid_predictions))

## 4.5. XGBOOST

In [ ]:
import xgboost
from sklearn.metrics import log_loss
xgb = xgboost.XGBClassifier(learning_rate=0.1,
                                max_depth=20,
                                min_child_weight=30,
                                n_estimators=20)
xgb.fit(X_train, y_train)
eval_set=[(X_test, y_test)]
predictions = xgb.predict(X_test)
print(log_loss(y_test, predictions))
print(confusion_matrix(y_test,predictions))
print(classification_report(y_test, predictions))
print(accuracy_score(y_test, predictions))

## 4.6.Artifial Neural Networks: 

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2
ann = Sequential()
ann.add(Dense(units = 50,activation="relu", kernel_regularizer=l2(0.001)))
ann.add(Dropout(0.2))
ann.add(Dense(units = 100, activation="relu",kernel_regularizer=l2(0.001)))
ann.add(Dropout(0.2))
ann.add(Dense(units = 200, activation="relu",kernel_regularizer=l2(0.001)))
ann.add(Dense(1,activation="sigmoid")) 

ann.compile(optimizer = "adam", loss="binary_crossentropy",metrics=["accuracy"])
callback=EarlyStopping(monitor="val_loss", patience=2)
history = ann.fit(x = X_train, y= y_train, validation_data=(X_test,y_test), batch_size=16, epochs=25,callbacks=[callback])